# Milestone 1 — NLP Foundation & Semantic Similarity (Baseline)

**Project:** Smart MCQ Solver Challenge  
**Model Type:** Built from scratch (no pretrained weights)  
**Approach:** TF-IDF and Word2Vec embeddings + Cosine Similarity  
**Metric:** MAP@3 (Mean Average Precision at 3)

In this milestone, we build two baseline models that use traditional NLP techniques to rank MCQ options by their textual similarity to the question prompt. These serve as our "model from scratch" requirement for the project.

**Pipeline:**
1. Load and explore the dataset
2. Clean and preprocess text
3. Generate TF-IDF embeddings → rank by cosine similarity
4. Generate Word2Vec embeddings → rank by cosine similarity
5. Evaluate using MAP@3
6. Log results to Weights & Biases
7. Submit baseline predictions to Kaggle

In [1]:
!pip install numpy pandas scikit-learn gensim nltk wandb -q

import numpy as np
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
import wandb
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Dataset

The competition dataset contains MCQ-style questions. Each row has:
- `id` — unique question identifier
- `prompt` — the question text
- `A`, `B`, `C`, `D`, `E` — five answer option texts
- `answer` — the correct answer label (only in train set)

We load both `train.csv` (for building and evaluating models) and `test.csv` (for Kaggle submission).

In [2]:

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"\nFirst 3 rows:")
train_df.head(3)

Train shape: (2000, 8)
Test shape:  (500, 7)

Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

First 3 rows:


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


## 2. Exploratory Data Analysis

Before building any model, we need to understand:
- Are there missing values in any column?
- Is the answer distribution balanced across A–E? (If skewed, a naive model could just always predict the most common label)
- How long are the prompts? Short prompts may lack context for similarity matching.
- What do actual questions look like?

In [3]:
# Check for missing values
print("Missing values:")
print(train_df.isnull().sum())

# Answer distribution — is it balanced across A-E?
print(f"\nAnswer distribution:")
print(train_df['answer'].value_counts().sort_index())

# Prompt length statistics
train_df['prompt_len'] = train_df['prompt'].astype(str).apply(len)
print(f"\nPrompt length stats (characters):")
print(train_df['prompt_len'].describe())

# Sample a question to see the format
print(f"\n{'='*60}")
print("SAMPLE QUESTION:")
print(f"{'='*60}")
sample = train_df.iloc[0]
print(f"Prompt: {sample['prompt'][:200]}...")
for opt in ['A', 'B', 'C', 'D', 'E']:
    marker = " ✓" if sample['answer'] == opt else ""
    print(f"  {opt}: {str(sample[opt])[:100]}{marker}")
print(f"Correct: {sample['answer']}")

Missing values:
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer distribution:
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Prompt length stats (characters):
count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt_len, dtype: float64

SAMPLE QUESTION:
Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options....
  A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not ha
  B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relation ✓
  C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consci
  D: Martin Heidegger believes that the relatio

## 3. Text Preprocessing

Raw text contains noise — punctuation, inconsistent casing, extra whitespace — that can hurt embedding quality. We define two cleaning functions:

1. **`clean_text`** — lowercase, remove special characters, normalize whitespace. Used for Word2Vec (stopwords carry semantic meaning in context).
2. **`clean_text_no_stopwords`** — additionally removes stopwords like "the", "is", "of". Used for TF-IDF where stopwords dilute term importance.

In [4]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Clean and normalize text for embedding."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_no_stopwords(text):
    """Clean text and remove stopwords (better for TF-IDF)."""
    text = clean_text(text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

# Test both functions
sample_text = "What is the PRIMARY function of mitochondria in a cell?"
print(f"Original:      {sample_text}")
print(f"Cleaned:       {clean_text(sample_text)}")
print(f"No stopwords:  {clean_text_no_stopwords(sample_text)}")

Original:      What is the PRIMARY function of mitochondria in a cell?
Cleaned:       what is the primary function of mitochondria in a cell
No stopwords:  primary function mitochondria cell


## 4. Apply Cleaning to Dataset

We create cleaned versions of the `prompt` and all five option columns (`A`–`E`) for both train and test sets. The original columns are preserved for reference.

In [5]:
option_cols = ['A', 'B', 'C', 'D', 'E']

def clean_dataframe(df):
    """Apply text cleaning to prompt and all option columns."""
    df = df.copy()
    df['prompt_clean'] = df['prompt'].apply(clean_text)
    for col in option_cols:
        df[f'{col}_clean'] = df[col].apply(clean_text)
    return df

train_df = clean_dataframe(train_df)
test_df = clean_dataframe(test_df)

print("Cleaning done.")
print(f"Sample cleaned prompt: {train_df['prompt_clean'].iloc[0][:150]}...")

Cleaning done.
Sample cleaned prompt: pick the best possible answer what is martin heidegger s view on the relationship between time and human existence among the listed options...


## 5. MAP@3 — Evaluation Metric

**MAP@3 (Mean Average Precision at 3)** is our competition metric.

For each question, we predict the top 3 most likely answers in ranked order. The scoring:
- Correct answer at **position 1** → score = **1.0**
- Correct answer at **position 2** → score = **0.5**
- Correct answer at **position 3** → score = **0.333**
- Correct answer **not in top 3** → score = **0.0**

**MAP@3** = average of all individual scores across all questions.

We also implement a detailed version that breaks down performance by position, which helps with error analysis later.

In [6]:
def ap_at_3(true_label, predicted_labels):
    """Average Precision at 3 for a single question."""
    for i, pred in enumerate(predicted_labels[:3]):
        if pred.strip().upper() == true_label.strip().upper():
            return 1.0 / (i + 1)
    return 0.0

def map_at_3(true_labels, predicted_labels):
    """Mean Average Precision at 3 across all questions."""
    scores = [ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)]
    return np.mean(scores)

def map_at_3_detailed(true_labels, predicted_labels):
    """MAP@3 with full breakdown for error analysis."""
    scores = [ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)]
    n = len(scores)
    return {
        'map3': np.mean(scores),
        'correct_at_1': sum(1 for s in scores if s == 1.0),
        'correct_at_2': sum(1 for s in scores if s == 0.5),
        'correct_at_3': sum(1 for s in scores if abs(s - 1/3) < 0.01),
        'missed': sum(1 for s in scores if s == 0.0),
        'total': n,
        'top1_acc': sum(1 for s in scores if s == 1.0) / n,
        'top3_acc': sum(1 for s in scores if s > 0) / n,
    }

# Sanity test
test_true = ["A", "C", "B"]
test_pred = [["A", "B", "C"], ["B", "C", "D"], ["D", "E", "A"]]
print(f"Sanity check MAP@3: {map_at_3(test_true, test_pred):.4f}")
# Expected: (1.0 + 0.5 + 0.0) / 3 = 0.5

Sanity check MAP@3: 0.5000


## 6. Model 1 — TF-IDF + Cosine Similarity

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts text into sparse numerical vectors based on word importance:
- **TF** — how often a word appears in a document
- **IDF** — penalizes words that appear in many documents (common words get lower weight)

**Our approach:**
1. Fit a TF-IDF vectorizer on ALL text (prompts + options from train + test)
2. For each question, transform the prompt and all 5 options into TF-IDF vectors
3. Compute cosine similarity between the prompt vector and each option vector
4. Rank options by similarity (highest = most likely answer)
5. Take top 3 as our prediction

**Why this is a valid "from scratch" model:** We're not using any pretrained weights — the TF-IDF vocabulary is learned directly from our competition corpus.

In [7]:
# Collect all text to fit the vectorizer on the full corpus
all_texts = []
for df in [train_df, test_df]:
    all_texts.extend(df['prompt_clean'].tolist())
    for col in option_cols:
        all_texts.extend(df[f'{col}_clean'].tolist())

# Fit TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=50000,     # limit vocabulary size
    ngram_range=(1, 2),     # use unigrams + bigrams
    min_df=2,               # ignore very rare terms
    max_df=0.95,            # ignore terms in >95% of docs
    sublinear_tf=True       # apply log normalization to term frequencies
)
tfidf.fit(all_texts)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Sample features: {list(tfidf.vocabulary_.keys())[:10]}")

Vocabulary size: 12592
Sample features: ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heidegger', 'view']


### 6.1 Prediction & Evaluation

For each question, we compute `cosine_similarity(prompt_vector, option_vector)` for all 5 options, sort descending, and take the top 3 labels as our prediction.

**Cosine similarity** measures the angle between two vectors — it ranges from -1 (opposite) to 1 (identical direction). Higher similarity means the option text is more "about the same thing" as the prompt.

In [8]:
def predict_tfidf(df, vectorizer):
    """
    Predict top-3 answers using TF-IDF cosine similarity.
    For each row, rank all 5 options by similarity to the prompt.
    """
    predictions = []
    prompt_vectors = vectorizer.transform(df['prompt_clean'].tolist())

    for idx in range(len(df)):
        prompt_vec = prompt_vectors[idx]

        similarities = {}
        for opt in option_cols:
            opt_vec = vectorizer.transform([df[f'{opt}_clean'].iloc[idx]])
            sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
            similarities[opt] = sim

        ranked = sorted(similarities, key=similarities.get, reverse=True)
        predictions.append(ranked[:3])

    return predictions

# Run predictions on training set
print("Running TF-IDF predictions on train set...")
train_preds_tfidf = predict_tfidf(train_df, tfidf)

# Evaluate
results_tfidf = map_at_3_detailed(train_df['answer'].tolist(), train_preds_tfidf)

print(f"\n{'='*50}")
print(f"TF-IDF Baseline Results")
print(f"{'='*50}")
print(f"  MAP@3:          {results_tfidf['map3']:.4f}")
print(f"  Top-1 Accuracy: {results_tfidf['top1_acc']:.2%}")
print(f"  Top-3 Accuracy: {results_tfidf['top3_acc']:.2%}")
print(f"  Correct at #1:  {results_tfidf['correct_at_1']}/{results_tfidf['total']}")
print(f"  Correct at #2:  {results_tfidf['correct_at_2']}/{results_tfidf['total']}")
print(f"  Correct at #3:  {results_tfidf['correct_at_3']}/{results_tfidf['total']}")
print(f"  Missed:         {results_tfidf['missed']}/{results_tfidf['total']}")

Running TF-IDF predictions on train set...

TF-IDF Baseline Results
  MAP@3:          0.2898
  Top-1 Accuracy: 13.60%
  Top-3 Accuracy: 49.65%
  Correct at #1:  272/2000
  Correct at #2:  403/2000
  Correct at #3:  318/2000
  Missed:         1007/2000


## 7. Model 2 — Word2Vec + Cosine Similarity

**Word2Vec** learns dense word embeddings by training a shallow neural network on the corpus. Unlike TF-IDF (which produces sparse, high-dimensional vectors based on word counts), Word2Vec produces dense, low-dimensional vectors that capture semantic relationships.

**Our approach:**
1. Tokenize all text and train a Word2Vec model using skip-gram
2. To represent a full sentence/document, we **average** all its word vectors (simple but effective baseline)
3. Compute cosine similarity between averaged prompt vector and averaged option vectors
4. Rank and predict top 3

**Key parameters:**
- `vector_size=100` — each word is represented as a 100-dimensional vector
- `window=5` — context window of 5 words
- `sg=1` — skip-gram architecture (generally better for smaller datasets than CBOW)
- `epochs=20` — number of training passes over the corpus

In [9]:
# Tokenize all text for Word2Vec training
tokenized_texts = [text.split() for text in all_texts if text.strip()]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=20,
    sg=1
)

print(f"Word2Vec vocabulary: {len(w2v_model.wv)} words")
print(f"Vector size: {w2v_model.wv.vector_size}")

Word2Vec vocabulary: 2973 words
Vector size: 100


### 7.1 Prediction & Evaluation

To get a document-level embedding from Word2Vec, we average the word vectors of all words in the text. If a word isn't in the vocabulary, it's skipped. If no words are found, we return a zero vector.

**Limitation:** Averaging loses word order information. "Dog bites man" and "Man bites dog" would get the same vector. This is a known weakness we'll address with transformers in Milestone 2.

In [10]:
def text_to_w2v_vector(text, model):
    """Average word vectors for all words in text."""
    words = text.split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if not vectors:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

def predict_w2v(df, model):
    """Predict top-3 answers using Word2Vec cosine similarity."""
    predictions = []

    for idx in range(len(df)):
        prompt_vec = text_to_w2v_vector(df['prompt_clean'].iloc[idx], model)
        prompt_vec = prompt_vec.reshape(1, -1)

        similarities = {}
        for opt in option_cols:
            opt_vec = text_to_w2v_vector(df[f'{opt}_clean'].iloc[idx], model)
            opt_vec = opt_vec.reshape(1, -1)

            if np.all(prompt_vec == 0) or np.all(opt_vec == 0):
                similarities[opt] = 0.0
            else:
                similarities[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]

        ranked = sorted(similarities, key=similarities.get, reverse=True)
        predictions.append(ranked[:3])

    return predictions

# Run predictions
print("Running Word2Vec predictions on train set...")
train_preds_w2v = predict_w2v(train_df, w2v_model)

# Evaluate
results_w2v = map_at_3_detailed(train_df['answer'].tolist(), train_preds_w2v)

print(f"\n{'='*50}")
print(f"Word2Vec Baseline Results")
print(f"{'='*50}")
print(f"  MAP@3:          {results_w2v['map3']:.4f}")
print(f"  Top-1 Accuracy: {results_w2v['top1_acc']:.2%}")
print(f"  Top-3 Accuracy: {results_w2v['top3_acc']:.2%}")
print(f"  Correct at #1:  {results_w2v['correct_at_1']}/{results_w2v['total']}")
print(f"  Missed:         {results_w2v['missed']}/{results_w2v['total']}")

Running Word2Vec predictions on train set...

Word2Vec Baseline Results
  MAP@3:          0.3349
  Top-1 Accuracy: 18.35%
  Top-3 Accuracy: 55.20%
  Correct at #1:  367/2000
  Missed:         896/2000


## 8. Model Comparison

Let's compare both baselines side by side. This comparison will also be logged to W&B for tracking across future milestones.

| Aspect | TF-IDF | Word2Vec |
|--------|--------|----------|
| Vector type | Sparse, high-dimensional | Dense, 100-dimensional |
| Captures | Exact word overlap | Semantic word relationships |
| Word order | Ignored | Ignored (due to averaging) |
| Training | No training (statistical) | Trained on corpus (neural) |

In [11]:
comparison = pd.DataFrame({
    'Metric': ['MAP@3', 'Top-1 Accuracy', 'Top-3 Accuracy', 'Missed Questions'],
    'TF-IDF': [
        f"{results_tfidf['map3']:.4f}",
        f"{results_tfidf['top1_acc']:.2%}",
        f"{results_tfidf['top3_acc']:.2%}",
        f"{results_tfidf['missed']}/{results_tfidf['total']}"
    ],
    'Word2Vec': [
        f"{results_w2v['map3']:.4f}",
        f"{results_w2v['top1_acc']:.2%}",
        f"{results_w2v['top3_acc']:.2%}",
        f"{results_w2v['missed']}/{results_w2v['total']}"
    ]
})
print(comparison.to_string(index=False))

          Metric    TF-IDF Word2Vec
           MAP@3    0.2898   0.3349
  Top-1 Accuracy    13.60%   18.35%
  Top-3 Accuracy    49.65%   55.20%
Missed Questions 1007/2000 896/2000


## 9. Experiment Tracking — Weights & Biases

We log both model runs to W&B so we can:
- Compare them visually on the W&B dashboard
- Track improvement across future milestones
- Show experiment history during viva

Each run logs: model type, MAP@3, top-1/top-3 accuracy, and model-specific hyperparameters.

In [12]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [13]:
# --- Run 1: TF-IDF ---
wandb.init(
    project="22f3002548-t22026",
    name="milestone1-tfidf-baseline",
    tags=["milestone1", "baseline", "tfidf"]
)
wandb.log({
    "model": "tfidf_cosine",
    "map3": results_tfidf['map3'],
    "top1_accuracy": results_tfidf['top1_acc'],
    "top3_accuracy": results_tfidf['top3_acc'],
    "missed": results_tfidf['missed'],
    "vocab_size": len(tfidf.vocabulary_),
    "ngram_range": "1-2",
    "max_features": 50000,
})
wandb.finish()

# --- Run 2: Word2Vec ---
wandb.init(
    project="YourRollNo-t22026",
    name="milestone1-word2vec-baseline",
    tags=["milestone1", "baseline", "word2vec"]
)
wandb.log({
    "model": "word2vec_cosine",
    "map3": results_w2v['map3'],
    "top1_accuracy": results_w2v['top1_acc'],
    "top3_accuracy": results_w2v['top3_acc'],
    "missed": results_w2v['missed'],
    "vector_size": w2v_model.wv.vector_size,
    "window": 5,
    "epochs": 20,
})
wandb.finish()

print("Both runs logged to W&B!")

wandb: setting up run lmnfj1hf
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260629_152934-lmnfj1hf
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run milestone1-tfidf-baseline
wandb: ⭐️ View project at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026
wandb: 🚀 View run at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026/runs/lmnfj1hf
wandb: updating run metadata; uploading summary
wandb: updating run metadata
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb:          map3 ▁
wandb:  max_features ▁
wandb:        missed ▁
wandb: top1_accuracy ▁
wandb: top3_accuracy ▁
wandb:    vocab_size ▁
wandb: 
wandb: Run summary:
wandb:          map3 0.28975
wandb:  max_features 50000
wandb:        missed 1007
wandb:         model tfidf_cosine
wandb:   ngram_range 1-2
wandb: top1_accuracy 0.136
wandb: top3_accuracy 0.4965
wandb:    vocab_size 12592
wandb: 
wandb: 🚀 V

Both runs logged to W&B!


## 10. Generate Kaggle Submission

We use whichever model scored higher on the training set to generate predictions for the test set. The submission format requires:

In [14]:
# Pick the better model
best_model = "tfidf" if results_tfidf['map3'] >= results_w2v['map3'] else "w2v"
print(f"Using {best_model} for submission (higher MAP@3 on train)")

# Generate predictions on test set
if best_model == "tfidf":
    test_preds = predict_tfidf(test_df, tfidf)
else:
    test_preds = predict_w2v(test_df, w2v_model)

# Format as submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': [' '.join(pred) for pred in test_preds]
})

print(f"\nSubmission shape: {submission.shape}")
print(f"\nFirst 5 rows:")
print(submission.head())

# Save
submission.to_csv('submission.csv', index=False)
print(f"\nSaved to submission.csv — ready to submit on Kaggle!")

Using w2v for submission (higher MAP@3 on train)

Submission shape: (500, 2)

First 5 rows:
   id prediction
0   1      C A B
1   2      B D C
2   3      D A C
3   4      A E C
4   5      B C A

Saved to submission.csv — ready to submit on Kaggle!


## 11. Error Analysis

Understanding *where* and *why* the model fails is crucial for:
- Improving the model in later milestones
- Writing a strong report (error analysis section)
- Answering viva questions ("What are the limitations of your baseline?")

We look at questions the model missed entirely (AP@3 = 0) and try to identify patterns — are they reasoning-heavy? Do they require external knowledge? Are the options too semantically similar?

In [15]:
# Attach scores to training data
train_scores = [ap_at_3(t, p) for t, p in zip(train_df['answer'].tolist(), train_preds_tfidf)]
train_df['ap3_score'] = train_scores

# Missed questions
missed_df = train_df[train_df['ap3_score'] == 0.0]
print(f"Questions missed entirely: {len(missed_df)}/{len(train_df)}")

# Show examples of failures
print(f"\nSample missed questions:")
print("=" * 60)

for i, (_, row) in enumerate(missed_df.head(3).iterrows()):
    print(f"\nQ{i+1}: {str(row['prompt'])[:150]}...")
    print(f"  Correct: {row['answer']} = {str(row[row['answer']])[:80]}")
    idx = row.name
    pred = train_preds_tfidf[idx]
    print(f"  Predicted: {' '.join(pred)}")
    print(f"  Top pick: {pred[0]} = {str(row[pred[0]])[:80]}")

# Score distribution
print(f"\n{'='*60}")
print("Score distribution:")
print(f"  Score 1.000 (correct at #1): {results_tfidf['correct_at_1']}")
print(f"  Score 0.500 (correct at #2): {results_tfidf['correct_at_2']}")
print(f"  Score 0.333 (correct at #3): {results_tfidf['correct_at_3']}")
print(f"  Score 0.000 (missed):        {results_tfidf['missed']}")

Questions missed entirely: 1007/2000

Sample missed questions:

Q1: Identify the correct statement: What is the concept of simultaneity in Einstein's book, Relativity? carefully....
  Correct: A = Simultaneity is relative, meaning that two events that appear simultaneous to an
  Predicted: D E B
  Top pick: D = Simultaneity is a concept that applies only to Newtonian theories and not to rel

Q2: Which of the following is correct? What is a "coffee ring" in physics? among the listed options....
  Correct: E = A pattern left by a particle-laden liquid after it evaporates, named for the cha
  Predicted: A C D
  Top pick: A = A type of coffee that is made by boiling coffee grounds in water.

Q3: Select the most accurate option: What is the Liouville density? among the listed options....
  Correct: A = The Liouville density is a probability distribution that specifies the probabili
  Predicted: C E B
  Top pick: C = The Liouville density is a bounded probability distribution that is a conve